# Обучение классификатора овощей (трек A)

Перед запуском:
1. Панель справа **Input → Add Input**: датасет `vegetable-dataset` (автор preethi866).
2. **Settings → Accelerator**: GPU T4 x2.
3. **Settings → Internet** должен быть включён (в меню пункт называется «Turn off internet»): он нужен для `git clone` и весов ImageNet.

In [ ]:
REPO = "https://github.com/Abylcor/vegetable-classifier.git"

%cd /kaggle/working
!rm -rf vegetable-classifier && git clone -q {REPO}
%cd vegetable-classifier
!ls
!nvidia-smi -L

## Проверка разбиения на Kaggle (воспроизводимость)
Строим разбиение заново из файлов датасета и сравниваем с `splits.csv` из репозитория. Ожидается `совпадает: 1.0`.

Нужен scikit-learn 1.8 или новее: в более старых версиях (на Kaggle стоит такая) `StratifiedGroupKFold` с `shuffle=True` перемешивает группы с ошибкой и даёт другое, плохо стратифицированное разбиение. Поэтому сначала ставим новую версию.

In [ ]:
!pip install -q "scikit-learn>=1.8"
!python make_splits.py --data /kaggle/input --out splits_kaggle.csv
import pandas as pd
a, b = pd.read_csv("splits.csv"), pd.read_csv("splits_kaggle.csv")
key = lambda d: d["label"] + "/" + d["path"].str.split("/").str[-1]
m = a.assign(k=key(a)).merge(b.assign(k=key(b)), on="k")
print("строк:", len(a), len(b), "| сопоставлено:", len(m), "| совпадает:", (m.fold_x == m.fold_y).mean())

## Наше разбиение: 2 фолда по видео

In [ ]:
!python train.py --data /kaggle/input --splits splits.csv --test-fold 0 --out runs/fold0

In [ ]:
!python train.py --data /kaggle/input --splits splits.csv --test-fold 1 --out runs/fold1

## Итог по обоим фолдам (out-of-fold)
Каждое из 3879 фото предсказано моделью, которая не видела его видео.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, ".")
from train import compute_metrics, print_metrics
from common import CLASSES
oof = pd.concat([pd.read_csv("runs/fold0/preds.csv"), pd.read_csv("runs/fold1/preds.csv")], ignore_index=True)
idx = {c: i for i, c in enumerate(CLASSES)}
print("фото:", len(oof), "| уникальных:", oof.path.nunique())
print_metrics(compute_metrics(oof.label.map(idx).tolist(), oof.pred.map(idx).tolist(), len(CLASSES)))
oof.to_csv("runs/oof_preds.csv", index=False)

## Для сравнения: авторское разбиение final_dataset
Учим на авторском train, проверяем на авторском test. Эта цифра нужна только для отчёта: она показывает, насколько утечка завышает метрику.

In [ ]:
!python train.py --data /kaggle/input --splits author_split.csv --train-folds train --test-fold test --out runs/author

## Архив с результатами (без весов)
Скачайте `/kaggle/working/results.zip` в панели **Output** справа до закрытия сессии: файлы сессии временные.

In [ ]:
!cd /kaggle/working/vegetable-classifier && zip -qr /kaggle/working/results.zip runs -x "*.pt"
!ls -la /kaggle/working/results.zip